# Projekt z przedmiotu *Eksploracja Danych*

## Etap 2:  Przygotowanie danych + Modelowanie

### Analizowany zbiór danych: **Brewer's Friend Beer Recipes**

#### Autorzy:
- Anna Sztukowska 188803
- Michał Sugalski 193290
- Lucjan Gackowski 193150


#### Ogólny opis zbioru

Zbiór **Brewer's Friend Beer Recipes** zawiera dane dotyczące domowych receptur piwa udostępnianych przez użytkowników platformy Brewer's Friend – narzędzia wspierającego amatorskich i półprofesjonalnych piwowarów. Każdy wiersz odpowiada jednej recepturze i zawiera ogólne parametry techniczne związane z procesem warzenia.
Dane obejmują szeroki zakres ogólnych parametrów warzenia, takich jak styl piwa, zawartość alkoholu (`ABV`), poziom goryczki (`IBU`), kolor (`SRM`), gęstość początkowa (`OG`) i końcowa (`FG`), metoda warzenia (np. all grain, extract), objętości na różnych etapach produkcji, a także temperatury fermentacji.
Dane mają postać numeryczną lub kategoryczną i mogą służyć do analizy trendów, porównań stylów piwa, klasteryzacji receptur lub budowy modeli predykcyjnych opartych na parametrach fizykochemicznych trunku.

#### Charakterystyka zbioru danych
- **Pochodzenie:** Dane zostały zebrane z platformy Brewer's Friend i udostępnione na Kaggle przez użytkownika jtrofe.
- **Format:** `.csv`
- **Liczba przykładów:** 73 861 receptur piwa
- **Liczba atrybutów:** 23 kolumny opisujące właściwości każdej receptury
- **Struktura:** Zbiór składa się z dwóch plików:
  - `recipeData.csv` – główny zbiór zawierający informacje o recepturach piwa
  - `styleData.csv` – uzupełniający zbiór zawierający opisy stylów piwa

#### Określenie celu eksploracji i kryteriów sukcesu

Celem eksploracji jest klasyfikacja stylu piwa na podstawie jego właściwości fizykochemicznych, takich jak zawartość alkoholu `(ABV)`, goryczka `(IBU)`, gęstości `(OG, FG)`, kolor `(SRM)` oraz metoda warzenia.
Docelowo rozwiązywanym problemem jest klasyfikacja wieloklasowa, ponieważ styl piwa przyjmuje wiele możliwych wartości nominalnych.
Dodatkowym celem jest zidentyfikowanie, które cechy mają największy wpływ na klasyfikację stylu piwa — będzie to realizowane m.in. przez analizę ważności cech (feature importance).

Najbardziej istotną metryką będzie `accuracy` (dokładność klasyfikacji), czyli stosunek poprawnie sklasyfikowanych próbek do ogólnej liczby próbek. Ze względu na potencjalną nierównowagę klas (niektóre style mogą występować znacznie częściej), zastosowana zostanie metryka pomocnicza - `Balanced Accuracy`.

`Balanced accuracy` - metryka obliczającą średnią arytmetyczną czułości (`recall`) dla każdej klasy, co zapewnia, że model jest oceniany sprawiedliwie, niezależnie od liczności poszczególnych stylów piwa. Dzięki temu unikniemy sytuacji, w której wysoka dokładność wynika wyłącznie z poprawnego klasyfikowania dominujących klas, podczas gdy rzadkie style są ignorowane.

Dodatkowo zastosowane zostaną następujące metryki:

**Macro F1-score** to średnia arytmetyczna wartości F1 obliczonych osobno dla każdej klasy. Traktuje wszystkie klasy z równą wagą, niezależnie od ich liczności. Jest to szczególnie przydatna metryka przy niezbalansowanych danych, ponieważ zapobiega faworyzowaniu dominujących klas.

Wzory:

- F1-score dla pojedynczej klasy:

  $$
  F1 = 2 \cdot \frac{\text{precyzja} \cdot \text{czułość}}{\text{precyzja} + \text{czułość}}
  $$

- Precyzja (precision):

  $$
  \text{precyzja} = \frac{TP}{TP + FP}
  $$
  gdzie:
  - TP (True Positive) – liczba przypadków poprawnie zaklasyfikowanych jako pozytywne (np. poprawnie rozpoznany styl piwa),
  - FP (False Positive) – liczba przypadków błędnie zaklasyfikowanych jako pozytywne (np. piwo przypisane do danego stylu, chociaż nim nie jest).

- Czułość (recall):

  $$
  \text{czułość} = \frac{TP}{TP + FN}
  $$

- Macro F1-score:

  $$
  \text{macro F1-score} = \frac{1}{N} \sum_{i=1}^{N} F1_i
  $$

  Gdzie \( N \) to liczba klas.

**Confusion matrix**

Macierz pomyłek (`confusion matrix`) pozwala przeanalizować, które style piwa są najczęściej mylone między sobą.

Sukces zostanie osiągnięty, jeżeli:
- model osiągnie `accuracy` powyżej 60%,
- model osiągnie `balanced accuracy` powyżej 60%,
- model osiągnie `macro F1-score` powyżej 65%.


Przy wieloklasowym problemie klasyfikacyjnym i nieidealnie zbalansowanych danych będzie to oznaczać skuteczną eksplorację stylów piwa na podstawie parametrów technicznych.

#### Dyskusja kroków dalszego postępowania



##### Dobór działania eksploracji

Zgodnie z celem eksploracji, który został zdefiniowany w Raporcie 1, dążymy do klasyfikacji stylu piwa na podstawie jego właściwości fizykochemicznych oraz identyfikacji kluczowych cech determinujących dany styl. Analiza wstępna wykazała, że atrybut `StyleID` wykazuje słabą korelację liniową z pojedynczymi cechami, co sugeruje złożony, nieliniowy charakter problemu. W związku z tym, wybrano dwuetapowe podejście algorytmiczne.

Wstępna analiza danych ujawniła trzy kluczowe wyzwania, które muszą zostać zaadresowane w dalszych krokach:

- Znaczna liczba brakujących danych w niektórych kolumnach.
- Duża liczba klas (stylów piwa), z których wiele jest niedostatecznie reprezentowanych (problem niezbalansowanych klas).
- Obecność licznych wartości odstających, które mogą zakłócać działanie algorytmów eksploracyjnych.


##### Dobór algorytmu eksploracji
1. Klasteryzacja (Grupowanie stylów piwa)

Pierwszym krokiem będzie uproszczenie problemu poprzez zastosowanie algorytmu klasteryzacji. Zamiast klasyfikować ponad 170 indywidualnych stylów, co przy niezbalansowanym zbiorze jest zadaniem niezwykle trudnym, połączymy je w mniejsze, spójne merytorycznie grupy.

- Wybrany algorytm: K-średnich (K-Means)


Algorytm K-średnich jest metodą uczenia maszynowego bez nadzoru, której celem jest podział zbioru danych na z góry określoną liczbę klastrów - w naszym przypadku `6 klastrów`. Działa on iteracyjnie, grupując podobne do siebie punkty danych, minimalizując wariancję wewnątrz klastrów. W naszym projekcie zastosujemy go na kluczowych cechach fizykochemicznych (`IBU`, `ABV`, `Color`), aby zidentyfikować naturalne skupiska receptur, które dzielą podobne parametry.

Zastosowanie klasteryzacji K-średnich przed właściwą klasyfikacją przynosi kluczowe korzyści:

- Redukcja złożoności:

    Zmniejszenie liczby klas z ponad 170 znacząco upraszcza zadanie klasyfikacyjne i zwiększa szansę na uzyskanie modelu o wysokiej skuteczności.

- Obsługa niezbalansowanych klas:

    Rzadkie style piwa, które mają zbyt mało próbek do efektywnego uczenia, zostaną połączone z podobnymi, liczniejszymi stylami, tworząc bardziej zrównoważone grupy.

- Zwiększenie interpretowalności:

    Grupy opisowe (np. "Lager & Pilsner" czy "Stout & Porter") są bardziej intuicyjne i użyteczne z biznesowego punktu widzenia niż pojedyncze, często bardzo niszowe style. Wyniki klasteryzacji zostaną zweryfikowane przy użyciu wiedzy domenowej, np. w oparciu o wytyczne BJCP (Beer Judge Certification Program).

<br />

2. Klasyfikacja (Predykcja grupy stylów)

Po utworzeniu grup stylów, głównym zadaniem będzie zbudowanie modelu klasyfikacyjnego, który na podstawie cech receptury przypisze ją do odpowiedniej grupy.

- Wybrany algorytm: Las Losowy (Random Forest Classifier)


Las Losowy buduje wiele drzew decyzyjnych w procesie treningu, a ostateczną predykcję podejmuje na podstawie "głosowania" większości z nich. Jest to jeden z najskuteczniejszych i najbardziej uniwersalnych algorytmów klasyfikacyjnych.

Random Forest jest idealnym wyborem dla naszego problemu z kilku powodów:

- Wysoka skuteczność i odporność na przeuczenie: Dzięki agregacji wyników z wielu drzew, algorytm jest znacznie bardziej stabilny i dokładny niż pojedyncze drzewo decyzyjne, jednocześnie minimalizując ryzyko przeuczenia.

- Zdolność do modelowania nieliniowych zależności: Jak wskazano w Raporcie 1, proste zależności liniowe nie wystarczają do opisania stylu piwa. Lasy Losowe doskonale radzą sobie z wychwytywaniem złożonych interakcji między wieloma cechami.

- Wbudowana analiza ważności cech: Algorytm w naturalny sposób dostarcza miarę ważności każdej cechy (feature importance), co bezpośrednio realizuje nasz dodatkowy cel, jakim jest identyfikacja najważniejszych parametrów wpływających na styl piwa.

- Odporność na wartości odstające i skalowanie danych: Lasy Losowe są mniej wrażliwe na outliery, których obecność stwierdzono w Raporcie 1, i nie wymagają skomplikowanego skalowania cech.

<br />

Połączenie nienadzorowanej klasteryzacji K-średnich z nadzorowaną klasyfikacją za pomocą Lasu Losowego stanowi solidną i kompleksową strategię, która adresuje kluczowe wyzwania zidentyfikowane w naszym zbiorze danych.


##### Dobór metody testowania wyników

Aby zapewnić wiarygodną ocenę modelu klasyfikacyjnego, szczególnie w kontekście zidentyfikowanego problemu niezbalansowanych klas, konieczne jest zastosowanie wcześniej wymienionych metryk (`Accuracy`, `Balanced Accuracy`, `Macro F1-score`) oraz analizy macierzy pomyłek (`confusion matrix`).

Zbiór danych zawiera `73 861` rekordów, po oczyszczeniu brakujących danych, uzyskujemy mniej niż połowę tych wartości, pomimo tego, wydaje się to być wystarczającą liczbą do podziału
danych na zbiór testowy i treningowy. Przy tak dużej ilości wierszy, możemy założyć, że losowy rozkład wartości pomiędzy zbiór treningowy i testowy będzie wystarczająco reprezentatywny.

#### Przygotowanie danych



##### Dane brakujące i dane do ujednolicenia

Kluczowe statystyki braków (na podstawie pierwszego raportu):

|    Kolumna    | Braki | % Braków |                 Decyzja                  |
|:-------------:|:-----:|:--------:|:----------------------------------------:|
| PrimingMethod | 67101 | 90.8%    |                  Usunąć                  |
| PrimingAmount | 69087 | 93.5%    |                  Usunąć                  |
| PitchRate     | 39252 | 53.1%    |                  Usunąć                  |
| MashThickness | 29864 | 40.4%    | Usunąć wiersze z brakującymi wartościami |
| PrimaryTemp   | 22662 | 30.7%    | Usunąć wiersze z brakującymi wartościami |
| BoilGravity   | 2990  | 4.0%     | Usunąć wiersze z brakującymi wartościami |

Uzasadnienie decyzji:
- Kolumny z >50% braków usuwane ze względu na niemożność wiarygodnej imputacji
- `MashThickness`, `PrimaryTemp` i `BoilGravity` pozostają zachowane pomimo znaczących braków - usuwamy tylko wiersze z brakującymi wartościami, ponieważ te kolumny mogą zawierać istotne informacje dla klasyfikacji stylu piwa.





##### Zamiana na nominalne/numeryczne

W naszym zbiorze danych zidentyfikowano dwie kluczowe kolumny nominalne, które wymagają konwersji: `SugarScale` oraz `BrewMethod`.

Do ich transformacji zostanie zastosowana technika kodowania etykietami (Label Encoding) poprzez zdefiniowane mapowania.

Atrybut `SugarScale`:
   - `Specific Gravity` zostanie zakodowane jako 0
   - `Plato` zostanie zakodowane jako 1

Atrybut `BrewMethod`:
   - `All Grain` zostanie zakodowane jako 0.
   - `extract` zostanie zakodowane jako 1.
   - `Partial Mash` zostanie zakodowane jako 2.
   - `BIAB` zostanie zakodowane jako 3

Dla algorytmów opartych na drzewach decyzyjnych, takich jak wybrany w naszym projekcie Las Losowy, proste kodowanie etykietami jest wystarczające i akceptowalne.

##### Podzbiór danych
Z oryginalnego zbioru danych zostały całkowicie usunięte kolumny `PrimingMethod`, `PrimingAmount` oraz `PitchRate` ze względu na dużą ilość brakujących wartości. Następnie usunięte zostały wiersze, które nie zawierały wartości w jakiejkolwiek z pozostałych "wybrakowanych" kolumn (`MashThickness`, `PrimaryTemp`, `BoilGravity`). Przed wykorzystaniem modelu klasyfikującego tak oczyszczony podzbiór podzielony został na dwa kolejne: treningowy i testowy mające odpowiednio `25178` i `6295` wierszy.

##### Uzupełnienie danych
*Brak*


Przeanalizowaliśmy zachowanie modelu klasyfikującego w przypadku zastosowania różnych metod uzupełniania danych. Początkowo brakujące wartości próbowaliśmy wypełniać medianami atrybutów, natomiast nie poprawiło to jakości modelu. Podjęta została również próba zastosowania regresji (liniowej i lasu losowego) w celu uzupełnienia braków, ale również w żadnym stopniu nie wpłynęła na dokładność klasyfikacji.


#### Utworzenie modelu - Ręczne Grupowanie i Klasyfikacja - Michal
Pierwsza wersja procesu modelowania zakładała scalenie wartości atrybutu celu (`Style`) w bardziej ogólne grupy, jako że w oryginalnym zbiorze danych może on przyjmować jedną z 175 różnych wartości. W tym celu nie został użyty algorytm klasteryzacji. W zamian została utworzony słownik określający, który styl do jakiej z nowo powstałych klas należy. W ten sposób ilość możliwych klas spadła ze 175 do 21. Nowo powstałe klasy:
- Ale
- Barleywine
- Belgian
- Bitter
- Bock
- Cider/Perry
- Fruit/Spiced
- Hybrid Ale
- Hybrid Lager
- Lager
- Mead
- Mild Ale
- Pale Ale
- Porter & Stout
- Scotch Ale
- Smoked
- Sour Ale
- Specialty
- Spiced/Seasonal
- Strong Ale
- Wheat Beer

Po przeprowadzeniu grupowania po atrybucie celu tak powstały zbiór jest dzielony na dwa podzbiory: treningowy i testowy w proporcji 80/20 z zachowaniem losowego podziału. Model oceniany jest na podstawie zbioru testowego, co pozwala na zbadanie dokładności zaklasyfikowania receptur do konkretnych grup.

##### Wyniki klasyfikacji

**Ogólne metryki**
| Metryka           | Wartość|
|-------------------|--------|
| Balanced Accuracy | 0.3987 |
| Accuracy          | 0.6842 |

**Szczegółowe metryki**

| Metryka        | Precision | Recall | F1-score | Support |
|----------------|-----------|--------|----------|---------|
|           Ale  |      0.48 | 0.54   |  0.51    | 510     |
|    Barleywine  |      0.59 |  0.59  |    0.59  | 61      |
|       Belgian  |      0.57 |  0.59  |    0.58  | 573     |
|        Bitter  |      0.60 |  0.33  |    0.42  | 219     |
|          Bock  |      0.70 |  0.67  |    0.68  | 57      |
|    Cider/Perry |     1.00  |   0.33 |     0.50 | 6       |
|   Fruit/Spiced |      0.83 |   0.05 |     0.10 | 91      |
|     Hybrid Ale |      0.59 |    0.28|      0.38| 125     |
|   Hybrid Lager |      0.73 |    0.12|      0.21| 65      |
|          Lager |      0.79 |    0.68|      0.73| 571     |
|           Mead |      0.56 |    0.64|      0.60| 14      |
|       Mild Ale |      0.61 |    0.49|      0.54| 39      |
|       Pale Ale |      0.73 |    0.89|      0.80| 2372    |
| Porter & Stout |      0.80 |    0.90|      0.85| 724     |
|     Scotch Ale |      0.46 |    0.21|      0.29| 80      |
|         Smoked |      0.00 |   0.00 |     0.00 | 32      |
|       Sour Ale |      0.65 |   0.40 |     0.49 | 111     |
|      Specialty |      0.67 |   0.02 |     0.04 | 108     |
| Spiced/Seasonal|      0.00 |   0.00 |     0.00 | 43      |
|     Strong Ale |      0.18 |   0.03 |     0.05 | 68      |
|     Wheat Beer |      0.58 |   0.61 |     0.60 | 426     |

<br>

| Typ średniej | Precision | Recall | F1-score | Support |
|--------------|-----------|--------|----------|---------|
| Macro avg    | 0.58      | 0.40   |   0.43   |   6295  |
| Weighted avg | 0.67      | 0.68   |   0.66   |   6295  |

Dzięki wartości metryki `Balanced Accuracy` wynoszącej 39.87% można zauważyć, że model słabo radzi sobie z klasyfikacją mniej reprezentowanych klas, co może wynikać z zastosowanego grupowania, w którym klasy tworzone były na podstawie własnej wiedzy i domysłów, a nie na podstawie faktycznych wartości parametrów piw.


#### Utworzenie modelu - Klasteryzacja i Klasyfikacja

W ramach analizy danych przeprowadziliśmy proces modelowania składający się z dwóch głównych etapów: klasteryzacji przy użyciu algorytmu K-średnich oraz klasyfikacji z wykorzystaniem Random Forest.

Etap klasteryzacji rozpoczęliśmy od przygotowania danych, usuwając rekordy z brakującymi wartościami i standaryzując wybrane 14 cech charakterystycznych piw, w tym takie parametry jak ekstrakt początkowy (`OG`), ekstrakt końcowy (`FG`), zawartość alkoholu (`ABV`), goryczka (`IBU`) czy kolor (`Color`). Algorytm K-średnich uruchomiliśmy z domyślnymi parametrami, ustalając liczbę klastrów na 6 oraz ziarno losowości 42 dla zapewnienia powtarzalności wyników. W wyniku działania algorytmu każdemu piwu przypisaliśmy przynależność do jednego z klastrów:

- **Klaster 0 – "Indian Pale Ale"** \
  Ten klaster obejmuje piwa o bardzo wysokim poziomie goryczki (średnie IBU niemal 84). Zawartość alkoholu jest wyraźnie powyżej średniej (7.94%), natomiast kolor mieści się w średnim zakresie. Klaster skupia piwa mocne i intensywne, o silnym chmielowym charakterze.
- **Klaster 1 – "Mocniejsze Ale"** \
  Piwa w tym klastrze charakteryzują się średnim kolorem (12.83) i umiarkowaną goryczką (IBU 42.25) i umiarkowaną zawartością alkoholu (6.06%). Odpowiada piwom górnej fermentacji o nieco większej niż typowa zawartość alkoholu i o nieco ciemniejszym kolorze.
- **Klaster 2 – "Klasyczne Ale/Lagery"** \
  To jeden z bardziej "neutralnych" klastrów: jasny kolor (8.38), niska goryczka (34.48) i umiarkowany alkohol (5.45%). Klaster reprezentuje piwa o przeciętnych, standardowych parametrach. Jest to najbardziej typowy zestaw wartości wśród wszystkich klastrów.
- **Klaster 3 – "Portery, Stouty, Barley Wine"** \
  Cechą wyróżniającą tego klastra jest bardzo ciemny kolor (średnio 36.72), przy zachowaniu średniego poziomu goryczki (47.67) i stosunkowo wysokiego alkoholu (7.00%). Piwa z tej grupy są wyraźnie ciemniejsze od pozostałych i mają mocniejszy charakter.
- **Klaster 4 – "Lekkie Ale"** \
  Ten klaster ma duże zróżnicowanie koloru (średnia 11.58, ale duże odchylenie), niską goryczkę (33.69) i umiarkowaną zawartość alkoholu (6.18%). Piwa są lekkie, ale różnorodne pod względem barwy i stylu. Klaster jest mniej spójny niż inne, co sugeruje mieszankę różnych typów piw o łagodnym profilu.
- **Klaster 5 – "Sesyjne"** \
  Pod względem parametrów ten klaster przypomina klaster 2, ale ma większą zmienność danych. Średnia goryczka wynosi 37.08, kolor 11.59, a alkohol 6.00%, co wskazuje na piwa lekkie, ale niejednolite. Klaster zawiera piwa o szerokim zakresie cech, ale generalnie o niższej intensywności.

Nazwy klastrów nadaliśmy po przeanalizowaniu statystyk opisowych każdego z nich, uwzględniając charakterystyczne wartości takich parametrów jak kolor, goryczka czy zawartość alkoholu. Przy ich formułowaniu kierowaliśmy się zarówno własną wiedzą, jak i informacjami zaczerpniętymi z forów i społeczności piwowarskich, gdzie omawiane są typowe cechy i zakresy parametrów dla różnych stylów piw.

Do wizualizacji wyników klasteryzacji wykorzystano projekcję wybranych cech (`IBU`, `ABV`, `Color`) w celu zobrazowania rozdzielności klastrów.

##### Statystyki klastrów

**Klaster 0**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 10.07 | 8.21 | 5.78 | 0.00 | 50.00 |
| IBU   | 83.91 | 74.54 | 81.14 | 0.00 | 3409.30 |
| ABV   | 7.94  | 7.58 | 2.19 | 1.50 | 52.16 |

**Klaster 1**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 12.83 | 7.62 | 11.75 | 2.43 | 50.00 |
| IBU   | 42.25 | 36.39 | 25.94 | 0.00 | 215.67 |
| ABV   | 6.06  | 5.71 | 1.49 | 2.20 | 15.04 |

**Klaster 2**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 8.38  | 6.61 | 5.11 | 0.00 | 44.05 |
| IBU   | 34.48 | 31.42 | 18.19 | 0.00 | 197.72 |
| ABV   | 5.45  | 5.42 | 0.91 | 0.06 | 10.92 |

**Klaster 3**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 36.72 | 36.29 | 8.19 | 18.85 | 50.00 |
| IBU   | 47.67 | 40.41 | 25.73 | 0.00 | 239.69 |
| ABV   | 7.00  | 6.50 | 1.97 | 1.86 | 20.99 |

**Klaster 4**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 11.58 | 5.45 | 13.13 | 3.12 | 50.00 |
| IBU   | 33.69 | 31.82 | 17.72 | 5.36 | 70.80 |
| ABV   | 6.18  | 5.77 | 1.88 | 2.15 | 14.24 |

**Klaster 5**

| Cecha | Średnia | Mediana | Odchylenie | Min | Max |
|-------|---------|---------|-------------|------|------|
| Color | 11.59 | 6.21 | 11.39 | 0.00 | 50.00 |
| IBU   | 37.08 | 30.32 | 26.14 | 0.00 | 177.32 |
| ABV   | 6.00  | 5.85 | 1.49 | 0.00 | 11.09 |

Na podstawie analizy statystyk klastrów można wyróżnić kilka wyraźnych tendencji. Klastry różnią się przede wszystkim pod względem zawartości alkoholu (`ABV`), goryczki (`IBU`) oraz koloru (`Color`), co sugeruje, że są związane z różnymi stylami piwa. Część klastrów skupia piwa intensywne, mocne i wyraźnie chmielone, podczas gdy inne reprezentują łagodniejsze, lżejsze style. Widoczne są klastry o bardzo wysokiej goryczce (np. klaster 0), jak i takie o niskiej zawartości alkoholu i jasnej barwie (np. klaster 2). Ciekawym zjawiskiem jest też duże zróżnicowanie wewnętrzne w niektórych klastrach — np. w kolorze piwa — co może wskazywać na obecność różnych podstylów w ramach jednej grupy.

  
Wykresy przedstawiające poszczególne średnie wartości atrybutów `Color`, `IBU` oraz `ABV` w każdym z klastrów.

![Opis obrazka](plots/klaster_abv.png)

![Opis obrazka](plots/klaster_ibu.png)

![Opis obrazka](plots/klaster_color.png)


W celu wizualizacji klastrów zastosowaliśmy redukcję wymiarowości z użyciem analizy głównych składowych (PCA – Principal Component Analysis). PCA umożliwia przekształcenie danych o wysokiej liczbie cech (wielowymiarowych) do przestrzeni o mniejszej liczbie wymiarów — w tym przypadku do dwóch, co pozwala na przedstawienie danych w formie wykresu 2D.

Metoda ta polega na obliczeniu nowych zmiennych (głównych składowych), które są liniowymi kombinacjami oryginalnych cech i zachowują jak największą część wariancji danych. Pierwsza składowa (PC1) wyjaśnia największy możliwy rozrzut danych, a druga (PC2) kolejną największą, przy czym jest ortogonalna względem pierwszej. Dzięki temu punkty reprezentujące obserwacje mogą być rozłożone w sposób możliwie najlepiej ukazujący różnice między klastrami. Kolor każdego punktu odpowiada przynależności do jednego z klastrów wyznaczonych wcześniej przez algorytm KMeans.

Wynik prezentuje się następująco:

![Opis obrazka](plots/clusters_cut.png)

**Ogólna interpretacja wizualizacji**

Wykres ukazuje, że klastry nie są idealnie odseparowane – obserwujemy znaczne nakładanie się na siebie poszczególnych grup. Jest to kluczowy wniosek, który wizualnie potwierdza, że granice między stylami piwa są często płynne i nieostre. Mimo to, widoczne są wyraźne koncentracje i gradienty kolorów, co świadczy o tym, że proces klasteryzacji z powodzeniem uchwycił fundamentalne różnice w charakterystyce piw.

Po przypisaniu etykiet klastrowych, przeprowadzono klasyfikację przy użyciu modelu lasów losowych (Random Forest), który uczony był na pełnym zestawie 14 cech. Dane zostały podzielone na zbiór treningowy i testowy w proporcji 80/20, z zachowaniem losowego podziału dla zapewnienia reprezentatywności. W modelu ograniczono maksymalną głębokość drzew do 5 (`max_depth=5`), co pozwoliło zredukować ryzyko przeuczenia. Model został następnie oceniony na zbiorze testowym, co pozwoliło określić jego skuteczność w przewidywaniu przynależności piwa do jednego z wcześniej wyznaczonych klastrów.

##### Wyniki klasyfikacji

**Ogólne metryki**
| Metryka           | Wartość     |
|-------------------|-------------|
| Balanced Accuracy | 0.9005      |
| Accuracy          | 0.9299      |

**Szczegółowe metryki**
| Klasa | Precision | Recall | F1-score | Support |
|-------|-----------|--------|----------|---------|
| 0     | 0.87      | 0.79   | 0.83     | 1191    |
| 1     | 0.98      | 1.00   | 0.99     | 204     |
| 2     | 0.94      | 0.96   | 0.95     | 3878    |
| 3     | 0.95      | 0.96   | 0.95     | 969     |
| 4     | 1.00      | 0.78   | 0.88     | 9       |
| 5     | 0.96      | 0.92   | 0.94     | 73      |

| Typ średniej  | Precision | Recall | F1-score | Support |
|---------------|-----------|--------|----------|---------|
| Macro avg     | 0.95      | 0.90   | 0.92     | 6324    |
| Weighted avg  | 0.93      | 0.93   | 0.93     | 6324    |

 Model osiągnął wysoką skuteczność, co potwierdzają metryki oceny na zbiorze testowym:
- `Balanced Accuracy` wyniosła 90,0%, , co oznacza, że model dobrze radzi sobie z rozpoznawaniem klas, uwzględniając ich nierównomierne rozłożenie.
- `Accuracy` na poziomie 92,99% wskazuje na ogólnie bardzo dobrą jakość klasyfikacji, choć częściowo wynika z dominacji liczniejszych klas.

`Balanced accuracy` jest niższa niż `accuracy`, ponieważ uwzględnia nierównomierną liczbę próbek w poszczególnych klasach, traktując każdą klasę jednakowo, natomiast `accuracy` jest dominowana przez klasy o większej liczebności, co może zawyżać ogólną skuteczność modelu. Dzięki temu `balanced accuracy` lepiej odzwierciedla jakość klasyfikacji również dla mniejszych klas.

Analiza szczegółowych metryk pokazuje, że model bardzo dobrze klasyfikuje dominujące klasy, np. klasę 1 (precision i recall bliskie 100%) oraz klasę 2 (f1-score 0.95). Klasa 4, jako najmniej liczna (9 próbek), jest trudniejsza do prawidłowego przewidzenia – mimo bardzo wysokiej precision (1.00), recall spada do 0.78, co sugeruje możliwe problemy z uogólnieniem modelu w tej grupie.

Wysokie wartości średnich (`weighted avg`) – powyżej 93% – potwierdzają, że model skutecznie przypisuje próbki do odpowiednich klastrów, dobrze radząc sobie z różnorodnością w liczebności klas.


W celu dokładnej oceny skuteczności klasyfikatora wygenerowaliśmy macierz pomyłek, która prezentuje się następująco:

![Opis obrazka](plots/confusion_matrix.png)

Macierz pomyłek przedstawia liczbę poprawnych i błędnych klasyfikacji dla każdej klasy. Wartości na przekątnej oznaczają trafne przypisania, natomiast pozostałe komórki wskazują, gdzie model popełnił błąd, myląc jedną klasę z inną.

Model bardzo dobrze radzi sobie z klasyfikacją klas 1, 2 i 3. Klasa 1 została rozpoznana bezbłędnie (204 poprawne przypisania), klasa 2 z bardzo wysoką skutecznością (3739 trafień na 3878), a klasa 3 osiągnęła 926 trafień na 969 przypadków. Drobne pomyłki występują głównie między klasami 2 i 0 oraz 3 i 0, co może wskazywać na pewne podobieństwa cech między tymi grupami.

Klasa 0 została poprawnie sklasyfikowana w 938 przypadkach, jednak aż 225 próbek błędnie przypisano do klasy 2 oraz 28 do klasy 3. Jest to największe źródło błędów w całej macierzy i może sugerować trudności modelu w precyzyjnym rozdzieleniu tych klas.

W przypadku najmniejszych klas, model osiąga zadowalające wyniki — klasa 4 została poprawnie rozpoznana w 7 na 9 przypadków (2 błędnie przypisane do klasy 5), a klasa 5 została poprawnie sklasyfikowana w 67 przypadkach na 73, przy pojedynczych pomyłkach z innymi klasami.

Macierz pomyłek pokazuje, że model skutecznie rozpoznaje dominujące klasy i dobrze radzi sobie z mniejszymi, choć w ich przypadku występują sporadyczne błędy. Największy potencjał poprawy widoczny jest w lepszym rozdzielaniu klas 0 i 2.

##### Analiza ważności cech
Przedstawiony wykres obrazuje względną ważność cech w modelu klasyfikacyjnym, pokazując, które zmienne miały największy wpływ na decyzje lasu losowego:

![Opis obrazka](plots/feature_importance.png)

Analiza ważności cech wykazała, że największy wpływ na klasyfikację przynależności piw do klastrów miały cechy związane z kolorem (`Color`), ekstraktem początkowym (`OG`) oraz zawartością alkoholu (`ABV`). Te trzy cechy łącznie odpowiadają za ponad 60% ważności modelu, co sugeruje, że są one kluczowymi wskaźnikami różnicującymi style piwa w analizowanym zbiorze danych. Z kolei cechy takie jak metoda warzenia (`BrewMethod`) okazały się nieistotne dla modelu, mając ważność równą zeru.


#### Eksperymenty z modelem i zbiorem danych

**Testowanie metod imputacji oraz różnych algorytmów klasyfikacji przy manualnym grupowaniu stylów piw**

W celu znalezienia optymalnego modelu predykcyjnego przeprowadziliśmy serię eksperymentów, których głównym celem było porównanie skuteczności różnych algorytmów klasyfikacyjnych w połączeniu z wieloma strategiami obsługi brakujących danych. Jak zidentyfikowano w Raporcie 1, istotnym wyzwaniem w naszym zbiorze jest duża liczba braków w kluczowych kolumnach, co wymagało dogłębnej analizy wpływu różnych metod ich uzupełniania na finalną jakość predykcji.

Nasze eksperymenty skupiły się na systematycznym testowaniu następujących scenariuszy:

1. Porównanie strategii obsługi brakujących danych

Badaliśmy szerokie spektrum podejść, od najprostszych po zaawansowane techniki imputacji, aby ocenić, która z nich najlepiej zachowuje wartość informacyjną zbioru danych. Testowane metody to:

- `dropna` (usunięcie wierszy): Najbardziej rygorystyczne podejście, polegające na całkowitym usunięciu każdego rekordu zawierającego choćby jedną brakującą wartość.

- `simple_mean / simple_median` (imputacja statystyczna): Uzupełnienie braków średnią arytmetyczną lub medianą z danej kolumny.

- `knn` (imputacja metodą k-najbliższych sąsiadów): Wypełnienie brakującej wartości na podstawie wartości od k najbardziej podobnych rekordów w zbiorze.

- `iterative` (imputacja iteracyjna): Zaawansowana technika oparta na regresji (podobna do MICE), gdzie każda cecha z brakami jest modelowana jako funkcja pozostałych cech, a proces jest powtarzany iteracyjnie aż do uzyskania stabilnych wyników.

2. Porównanie algorytmów klasyfikacyjnych

Dla każdej z powyższych strategii przygotowania danych przetestowano szereg modeli klasyfikujących, aby sprawdzić, który z nich najlepiej radzi sobie ze specyfiką naszego problemu:

Modele zespołowe oparte na drzewach:
- `Random Forest`
- `Gradient Boosting`
- `HistGradientBoosting`

Klasyczne modele statystyczne:
- Regresja Logistyczna

Inne popularne algorytmy:
- Maszyny Wektorów Nośnych `(SVC)`
- k-Najbliższych Sąsiadów `(KNN)`,
- Pojedyncze Drzewo Decyzyjne oraz Naiwny Klasyfikator Bayesowski.

Wnioski z przeprowadzonych eksperymentów
Analiza wyników, zebranych w postaci metryk accuracy i macro F1-score, doprowadziła do kilku kluczowych i momentami zaskakujących spostrzeżeń:

Najlepsza strategia: Usunięcie brakujących danych `(dropna)`

Najwyższą skuteczność we wszystkich przeprowadzonych testach osiągnął model Random Forest wytrenowany na zbiorze danych, z którego po prostu usunięto wszystkie wiersze z brakującymi wartościami. Ta kombinacja uzyskała accuracy na poziomie `0.612` oraz F1-score równe `0.589`.

Wyniki przeprowadzonych eksperymentów:

| Metoda imputacji | Klasyfikator         | Accuracy | F1-score  |
|------------------|----------------------|----------|-----------|
|      dropna      |     RandomForest     | 0.612073 | 0.589128  |
| dropna           | GradientBoosting     | 0.597458 | 0.580891  |
| dropna           | HistGradientBoosting | 0.589833 | 0.570572  |
| simple_mean      | RandomForest         | 0.579062 | 0.551581  |
| knn              | RandomForest         | 0.578380 | 0.550524  |
| iterative        | RandomForest         | 0.577220 | 0.548768  |
| simple_mean      | GradientBoosting     | 0.574763 | 0.550711  |
| simple_median    | RandomForest         | 0.574490 | 0.547327  |
| dropna           | SVC                  | 0.573948 | 0.539541  |
| iterative        | GradientBoosting     | 0.573944 | 0.549718  |
| simple_median    | GradientBoosting     | 0.573807 | 0.549866  |
| knn              | GradientBoosting     | 0.571419 | 0.546873  |
| simple_mean      | HistGradientBoosting | 0.563844 | 0.536922  |
| simple_median    | HistGradientBoosting | 0.562137 | 0.536104  |
| iterative        | HistGradientBoosting | 0.560363 | 0.532690  |
| knn              | HistGradientBoosting | 0.557906 | 0.531369  |
| dropna           | LogisticRegression   | 0.540270 | 0.487471  |
| simple_median    | SVC                  | 0.536477 | 0.500335  |
| knn              | SVC                  | 0.534976 | 0.497670  |
| dropna           | KNN                  | 0.534710 | 0.517284  |
| iterative        | SVC                  | 0.534362 | 0.497116  |
| simple_mean      | SVC                  | 0.533816 | 0.496576  |
| simple_median    | KNN                  | 0.497782 | 0.478287  |
| knn              | KNN                  | 0.496758 | 0.476844  |
| iterative        | KNN                  | 0.496212 | 0.476507  |
| simple_mean      | KNN                  | 0.495189 | 0.475178  |
| knn              | LogisticRegression   | 0.485498 | 0.424022ś |
| iterative        | LogisticRegression   | 0.484747 | 0.424154  |
| simple_mean      | LogisticRegression   | 0.484542 | 0.424173  |
| simple_median    | LogisticRegression   | 0.481676 | 0.421160  |
| dropna           | DecisionTree         | 0.470850 | 0.472954  |
| simple_mean      | DecisionTree         | 0.437726 | 0.440390  |
| iterative        | DecisionTree         | 0.435201 | 0.438343  |
| simple_median    | DecisionTree         | 0.434109 | 0.436659  |
| knn              | DecisionTree         | 0.427353 | 0.431013  |
| simple_mean      | NaiveBayes           | 0.220023 | 0.263998  |
| knn              | NaiveBayes           | 0.210469 | 0.256586  |
| iterative        | NaiveBayes           | 0.204190 | 0.251684  |
| simple_median    | NaiveBayes           | 0.194499 | 0.246099  |
| dropna           | NaiveBayes           | 0.089913 | 0.114198  |


**Zmiana ilości klastrów w Klasteryzacji i ograniczenie modelu Klasyfikacji**

Przetestowano działanie modelu w warunkach silnego ograniczenia, ustawiając głębokość drzew na 5 przy 10 klastrach. Analiza macierzy pomyłek dla tego scenariusza ujawnia kluczowy problem: model jest niedouczony i niezdolny do poprawnego rozpoznania wszystkich klas, co było bardzo poważnym problemem.

Macierz pomyłek dla tego przypadku wygląda następująco:

![Opis obrazka](plots/confusion_matrix_depth5_clusters10.png)

Najbardziej alarmującym wynikiem, widocznym na macierzy, jest obecność zer na głównej przekątnej dla klastrów 6 i 7. Oznacza to, że model ani razu nie zidentyfikował poprawnie piwa z tych grup. W praktyce nauczył się je całkowicie ignorować, systematycznie klasyfikując ich próbki do innych, liczniejszych kategorii, głównie do klastra 2. Jest to skrajny przykład problemu, w którym zbyt prosty model, nie mogąc nauczyć się subtelnych różnic, uzyskuje bardzo niski poziom trafności.

#### Podsumowanie wyników




